# Part I: Matrices and transformations: solutions to the code exercises

Worked solutions to the code exercises from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/appendices/solutions-1.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/solutions-part-1.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Try each exercise yourself first. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

## Chapter 1: What is a matrix?

### Exercise 6

Create the $6 \times 6$ matrix with entries $a_{ij} = 1/(i + j - 1)$, and check that it is symmetric. (It is called the *Hilbert matrix*, and it returns in [the chapter](https://trgoodale.github.io/matrix-book/part-5/24-floating-point-conditioning.html) as a famously difficult matrix for computers.) In Python you may find `np.fromfunction` useful; in MATLAB, try `hilb(6)` and compare.

In [ ]:
import numpy as np

# Python counts from 0, so entry [i, j] corresponds to row i+1, column j+1
H = np.fromfunction(lambda i, j: 1 / (i + j + 1), (6, 6))
print(np.round(H, 4))
print(np.array_equal(H, H.T))

## Chapter 2: Adding and scaling matrices

### Exercise 6

The matrix `T` from [the chapter](https://trgoodale.github.io/matrix-book/part-1/01-matrices.html) holds four hourly readings from three temperature sensors. Subtract each sensor's average reading from its column, using broadcasting, and check that each column of the result adds up to zero (up to rounding).

In [ ]:
import numpy as np

T = np.array([[18.2, 19.1, 17.6],
              [19.5, 20.3, 18.4],
              [21.0, 21.8, 19.9],
              [22.4, 23.0, 21.2]])
D = T - T.mean(axis=0)       # the 1 x 3 row of means is broadcast down the rows
print(D)
print(D.sum(axis=0))         # zero, up to rounding

## Chapter 3: Matrix multiplication

### Exercise 7

For the road network $\mathbf{N}$ of Example 2, compute $\mathbf{N}^3$ and use it to find the number of three-step routes from town 1 back to town 1. List them. Why is the corresponding entry for town 4 zero?

In [ ]:
import numpy as np

N = np.array([[0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]])
print(np.linalg.matrix_power(N, 3))

The $(1, 1)$ entry is 2: the routes are 1 → 2 → 3 → 1 and 1 → 3 → 2 → 1, round the triangle in each direction. A three-step route that returns to its start must go round a triangle of roads. Town 4 has only one road, to town 3, so it is not on any triangle, and the $(4, 4)$ entry is 0.

## Chapter 4: Matrices as transformations

### Exercise 7

Apply the rotation $\mathbf{R}(\theta)$ to the F shape (corner coordinates are in the code for fig-transformation-gallery) for $\theta = 0°, 45°, 90°, \dots, 315°$, and plot all eight images on one set of axes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

F = np.array([[0, 0], [0.5, 0], [0.5, 0.85], [1.1, 0.85], [1.1, 1.3], [0.5, 1.3],
              [0.5, 1.75], [1.3, 1.75], [1.3, 2.2], [0, 2.2]]).T

fig, ax = plt.subplots(figsize=(5, 5))
for k in range(8):
    t = np.radians(45 * k)
    R = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    ax.fill(*(R @ F), alpha=0.25)
ax.set_aspect("equal"); ax.grid(alpha=0.3)
plt.show()

## Chapter 5: Composing transformations

### Exercise 7

Pick two random $2 \times 2$ matrices and check numerically that $(\mathbf{A}\mathbf{B})^{\mathsf{T}} = \mathbf{B}^{\mathsf{T}}\mathbf{A}^{\mathsf{T}}$, but that $\mathbf{A}\mathbf{B} \neq \mathbf{B}\mathbf{A}$. Then find, by experiment, a non-diagonal matrix that commutes with $\begin{pmatrix}2 & 1 \\ 1 & 2\end{pmatrix}$.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
A, B = rng.standard_normal((2, 2)), rng.standard_normal((2, 2))
print(np.allclose((A @ B).T, B.T @ A.T))     # True
print(np.allclose(A @ B, B @ A))             # False (almost always)

S = np.array([[2, 1], [1, 2]])
M = np.array([[1, 2], [2, 1]])               # a candidate
print(np.allclose(S @ M, M @ S))             # True

Any matrix of the form $\begin{pmatrix}a & b \\ b & a\end{pmatrix}$ commutes with $\begin{pmatrix}2 & 1 \\ 1 & 2\end{pmatrix}$. (These matrices all have the same invariant lines, $y = x$ and $y = -x$, which [the chapter](https://trgoodale.github.io/matrix-book/part-4/19-diagonalisation.html) explains.)

## Chapter 6: Determinants

### Exercise 8

Generate random $n \times n$ matrices with entries between $-1$ and $1$, and record $|\det\mathbf{A}|$ for $n = 2, 5, 10, 20, 50$. How does the typical size of the determinant change with $n$? Why is the size of a determinant a poor guide to whether a matrix is nearly singular?

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [2, 5, 10, 20, 50]:
    dets = [abs(np.linalg.det(rng.uniform(-1, 1, (n, n)))) for _ in range(200)]
    print(f"n = {n:2d}: median |det| = {np.median(dets):.3g}")

The typical size grows enormously with $n$, from about 0.3 to about $10^{19}$, although the matrices are no closer to or further from being singular. A determinant multiplies together $n$ scale factors, so it depends on the size of the entries and on $n$, not just on how close the matrix is to singular. The condition number ([the chapter](https://trgoodale.github.io/matrix-book/part-5/24-floating-point-conditioning.html)) measures nearness to singularity properly.

## Chapter 7: Inverse matrices

### Exercise 7

The *Hill cipher* is a simple secret code. It turns letters into numbers ($A = 0, B = 1, \dots, Z = 25$), multiplies pairs of them by a key matrix, and then replaces each result by its remainder after dividing by 26 ("reducing modulo 26"), so that it is a letter again. With key $\mathbf{K} = \begin{pmatrix}3 & 3 \\ 2 & 5\end{pmatrix}$, the message HELP becomes the vectors $(7, 4)^{\mathsf{T}}$ and $(11, 15)^{\mathsf{T}}$. Encrypt it. Then check that $\mathbf{K}^{-1}$ modulo 26 is $\begin{pmatrix}15 & 17 \\ 20 & 9\end{pmatrix}$, by showing that $\mathbf{K}$ times this matrix is $\mathbf{I}$ modulo 26, and use it to decrypt. ([the chapter](https://trgoodale.github.io/matrix-book/part-6/39-cryptography.html) explains how this inverse is found.)

In [ ]:
import numpy as np

K = np.array([[3, 3], [2, 5]])
Kinv = np.array([[15, 17], [20, 9]])
P = np.array([[7, 11], [4, 15]])          # HELP: one pair per column

C = (K @ P) % 26
print(C)                                  # (7, 8), (0, 19): HIAT
print((K @ Kinv) % 26)                    # the identity, modulo 26
print((Kinv @ C) % 26)                    # back to HELP

The ciphertext is HIAT.

## Chapter 8: Transformations in three dimensions

### Exercise 6

Aircraft orientation is often given as three angles: *yaw* $\psi$ (turning left or right, about the vertical $z$-axis), *pitch* $\theta$ (nose up or down, about $y$) and *roll* $\phi$ (tilting the wings, about $x$), applied in the aircraft's own frame, which gives the matrix $\mathbf{R}_z(\psi)\mathbf{R}_y(\theta)\mathbf{R}_x(\phi)$. Compute this, $\mathbf{R}_1$, for $\psi = 30°$, $\theta = 10°$, $\phi = 5°$, and compare it with the product in the reverse order, $\mathbf{R}_2$. By how many degrees do the two orientations differ? (*Hint*: the rotation angle of an orthogonal matrix $\mathbf{M}$ with determinant 1 satisfies $\operatorname{trace}\mathbf{M} = 1 + 2\cos\alpha$, where the trace is the sum of the diagonal entries; apply this to $\mathbf{M} = \mathbf{R}_1^{\mathsf{T}}\mathbf{R}_2$.)

In [ ]:
import numpy as np

def Rx(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def Ry(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def Rz(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

psi, theta, phi = np.radians([30, 10, 5])
R1 = Rz(psi) @ Ry(theta) @ Rx(phi)
R2 = Rx(phi) @ Ry(theta) @ Rz(psi)
M = R1.T @ R2
angle = np.degrees(np.arccos((np.trace(M) - 1) / 2))
print(f"The two orientations differ by {angle:.2f} degrees")

A difference of almost $6°$, from angles of only $5°$ to $30°$, is easily enough to matter in navigation.

## Chapter 9: Invariant points and lines

### Exercise 7

Use eigenvalues and eigenvectors to find the invariant lines of $\begin{pmatrix}1 & 2 \\ 3 & 2\end{pmatrix}$, and the stretch factor along each. Check your answer with eq-invariant-lines, and then with the app. What does the negative stretch factor mean for the points on that line?

In [ ]:
import numpy as np

A = np.array([[1, 2], [3, 2]])
values, vectors = np.linalg.eig(A)
values, vectors = values.real, vectors.real
for lam, v in zip(values, vectors.T):
    print(f"lambda = {lam:.4g}, slope {v[1] / v[0]:.4g}")

The invariant lines are $y = -x$ with $\lambda = -1$ and $y = \tfrac{3}{2}x$ with $\lambda = 4$. By hand, $b = 2$, $a - d = -1$, $c = 3$, so eq-invariant-lines is $2m^2 - m - 3 = 0$, that is $(2m - 3)(m + 1) = 0$, giving the same slopes. The stretch factors are $a + bm = 1 + 2 \times (-1) = -1$ and $1 + 2 \times \tfrac{3}{2} = 4$.

On $y = -x$, the stretch factor $-1$ means each point flips to the opposite side of the origin, at the same distance: $(1, -1)^{\mathsf{T}} \to (-1, 1)^{\mathsf{T}}$. The line lands on itself, but only the origin stays put.

## Part I: Matrices and transformations review

### Exercise 9

Make two random $4 \times 4$ matrices $\mathbf{A}$ and $\mathbf{B}$. Check numerically that $(\mathbf{A}\mathbf{B})^{-1} = \mathbf{B}^{-1}\mathbf{A}^{-1}$, and that this is *not* the same as $\mathbf{A}^{-1}\mathbf{B}^{-1}$. Explain the order using the idea of undoing two transformations.

*Uses [the chapter](https://trgoodale.github.io/matrix-book/part-1/07-inverses.html) and [the chapter](https://trgoodale.github.io/matrix-book/part-1/05-composition.html).*

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
A = rng.standard_normal((4, 4))
B = rng.standard_normal((4, 4))
inv = np.linalg.inv

print("(AB)^-1 = B^-1 A^-1:", np.allclose(inv(A @ B), inv(B) @ inv(A)))
print("(AB)^-1 = A^-1 B^-1:", np.allclose(inv(A @ B), inv(A) @ inv(B)))

$\mathbf{A}\mathbf{B}$ applies $\mathbf{B}$ first and then $\mathbf{A}$. To undo this, undo the last step first: apply $\mathbf{A}^{-1}$, then $\mathbf{B}^{-1}$, which is the matrix $\mathbf{B}^{-1}\mathbf{A}^{-1}$. It's like taking off shoes and socks: the shoes, which went on last, come off first.